# Detector reliability under corruption: Colab quickstart

Reproduces every table and figure of [dhrgam79/detector-reliability](https://github.com/dhrgam79/detector-reliability) from a fresh clone.

**Before running:** *Runtime → Change runtime type → T4 GPU*. A full run takes about 45 minutes; keep this tab open.


In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Clone and install (Colab already has PyTorch with CUDA)
%cd /content
!rm -rf /content/detector-reliability
!git clone -q https://github.com/dhrgam79/detector-reliability
%cd /content/detector-reliability
!git log --oneline -1
!pip install -q -e ".[dev]"

### Optional: keep data and detections on Google Drive

Skip this for a clean end-to-end reproduction. Use it if you expect to re-run: COCO images and cached detections then survive between Colab sessions, and a re-run only recomputes what changed.


In [ ]:
# 3. (optional) Uncomment to cache data and detections on Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# P = "/content/drive/MyDrive/detector-reliability"
# !mkdir -p "$P/data" "$P/cache"
# !ln -s "$P/data" data && ln -s "$P/cache" results/cache

In [ ]:
# 4. Run everything: data -> inference (27 conditions) -> metrics -> calibration -> figures
import time

t0 = time.time()
!python scripts/run_all.py 2>&1 | tail -40
print(f"Wall time: {(time.time() - t0) / 60:.1f} min")

In [ ]:
# 5. Figures
from IPython.display import Image, display

for f in ["fig1_mAP50_vs_severity", "fig2_dece_t0.3_before_after", "fig3_reliability_vru"]:
    display(Image(f"results/figures/{f}.png"))

In [ ]:
# 6. Key numbers: vulnerable road users at severity 5 vs clean (paired 95% CIs)
import pandas as pd

s = pd.read_csv("results/summary.csv")
vru = s[(s.scope == "vru") & (s.severity.isin([0, 5]))]
cols = [
    "corruption",
    "mAP50",
    "d_mAP50",
    "d_mAP50_lo",
    "d_mAP50_hi",
    "dece_t0.3",
    "d_dece_t0.3",
    "d_dece_t0.3_lo",
    "d_dece_t0.3_hi",
    "cal_gain_t0.3",
]
vru[cols].round(3)

In [ ]:
# 7. Runtime and calibration record of this run
from pathlib import Path

for f in ["results/run_info.json", "results/calibration.json"]:
    print(Path(f).read_text())

In [ ]:
# 8. Download the results (to commit them, or to compare with the repo)
from google.colab import files

for f in [
    "results/summary.csv",
    "results/reliability.csv",
    "results/calibration.json",
    "results/run_info.json",
]:
    files.download(f)